In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip install captum -q

In [ ]:
import os, gc, time, torch, numpy as np, pandas as pd
from PIL import Image
from io import BytesIO
from tqdm import tqdm
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
import torchvision.models as models
import torchvision.transforms as transforms
import torchvision.transforms.functional as TF
from captum.attr import IntegratedGradients, GradientShap, LayerGradCam

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(DEVICE, torch.cuda.get_device_name(0) if DEVICE == 'cuda' else '')

In [ ]:
DRIVE_BASE = '/content/drive/MyDrive/xai-stability-benchmark'

DATASETS = {
    'cifar10': {
        'data_path': os.path.join(DRIVE_BASE, 'datasets/cifar_sample.npz'),
        'result_dir': os.path.join(DRIVE_BASE, 'results_cifar10'),
        'type': 'npz',
        'num_classes': 1000,
    },
    'imagenet': {
        'data_path': os.path.join(DRIVE_BASE, 'datasets/imagenet_sample.parquet'),
        'result_dir': os.path.join(DRIVE_BASE, 'results_imagenet'),
        'type': 'parquet',
        'num_classes': 1000,
    },
    'coco': {
        'data_path': '/content/drive/MyDrive/xai-stability-data/coco/train2017',
        'result_dir': os.path.join(DRIVE_BASE, 'results_coco'),
        'type': 'directory',
        'num_classes': 80,
    },
}

MODEL_NAMES = ['resnet50', 'densenet121', 'convnext_tiny', 'vit_b_16']
PERTURBATIONS_TO_RUN = ['translation', 'brightness', 'jpeg']
BATCH_SIZE = 64
MAX_IMAGES = 10000
CHECKPOINT_EVERY = 512
NUM_WORKERS = 4

for cfg in DATASETS.values():
    os.makedirs(cfg['result_dir'], exist_ok=True)

In [ ]:
def get_ssim_spatial(attr1, attr2):
    def norm(x):
        b = x.shape[0]
        flat = x.view(b, -1)
        if x.dim() == 4:
            mn = flat.min(dim=1, keepdim=True)[0].view(b, 1, 1, 1)
            mx = flat.max(dim=1, keepdim=True)[0].view(b, 1, 1, 1)
        else:
            mn = flat.min(dim=1, keepdim=True)[0].view(b, 1, 1)
            mx = flat.max(dim=1, keepdim=True)[0].view(b, 1, 1)
        return (x - mn) / (mx - mn + 1e-8)
    a1, a2 = norm(attr1), norm(attr2)
    pool = nn.AvgPool2d(11, stride=1, padding=5)
    mu1, mu2 = pool(a1), pool(a2)
    sigma12 = pool(a1 * a2) - mu1 * mu2
    sigma1_sq = pool(a1 * a1) - mu1 * mu1
    sigma2_sq = pool(a2 * a2) - mu2 * mu2
    c1, c2 = 0.01**2, 0.03**2
    ssim_map = ((2*mu1*mu2+c1)*(2*sigma12+c2)) / ((mu1**2+mu2**2+c1)*(sigma1_sq+sigma2_sq+c2))
    return ssim_map.view(attr1.shape[0], -1).mean(dim=1)


def get_spearman_rank(attr1, attr2):
    b = attr1.shape[0]
    v1, v2 = attr1.view(b, -1), attr2.view(b, -1)
    r1 = torch.argsort(torch.argsort(v1, dim=1), dim=1).float()
    r2 = torch.argsort(torch.argsort(v2, dim=1), dim=1).float()
    mu1, mu2 = r1.mean(dim=1, keepdim=True), r2.mean(dim=1, keepdim=True)
    num = ((r1 - mu1) * (r2 - mu2)).sum(dim=1)
    den = torch.sqrt(((r1-mu1)**2).sum(dim=1) * ((r2-mu2)**2).sum(dim=1)) + 1e-8
    return (num / den + 1) / 2


def get_top_k_jaccard(attr1, attr2, k=100):
    b = attr1.shape[0]
    v1, v2 = attr1.view(b, -1), attr2.view(b, -1)
    _, idx1 = torch.topk(v1, k, dim=1)
    _, idx2 = torch.topk(v2, k, dim=1)
    jaccards = []
    for i in range(b):
        mask = torch.isin(idx1[i], idx2[i])
        intersection = mask.sum().float()
        union = 2 * k - intersection
        jaccards.append(intersection / union)
    return torch.tensor(jaccards, device=attr1.device)


def calculate_fass(attr1, attr2):
    return (get_ssim_spatial(attr1, attr2) + get_spearman_rank(attr1, attr2) + get_top_k_jaccard(attr1, attr2)) / 3

In [ ]:
def apply_perturbation(img, p_type, to_tensor, norm_tf):
    if p_type == 'rotation':
        return norm_tf(to_tensor(TF.rotate(img, 15)))
    elif p_type == 'noise':
        t = to_tensor(img)
        return norm_tf(torch.clamp(t + torch.randn_like(t) * 0.15, 0, 1))
    elif p_type == 'brightness':
        return norm_tf(to_tensor(TF.adjust_brightness(img, 1.5)))
    elif p_type == 'translation':
        return norm_tf(to_tensor(TF.affine(img, angle=0, translate=(20, 0), scale=1.0, shear=0)))
    elif p_type == 'jpeg':
        buf = BytesIO()
        img.save(buf, format='JPEG', quality=40)
        buf.seek(0)
        return norm_tf(to_tensor(Image.open(buf).convert('RGB')))


NORM_MEAN = [0.485, 0.456, 0.406]
NORM_STD = [0.229, 0.224, 0.225]


class FASSCIFAR10Dataset(Dataset):
    def __init__(self, images, p_type):
        self.images = images
        self.p_type = p_type
        self.base_tf = transforms.Compose([transforms.ToTensor(), transforms.Normalize(NORM_MEAN, NORM_STD)])
        self.norm_tf = transforms.Normalize(NORM_MEAN, NORM_STD)
    def __len__(self): return len(self.images)
    def __getitem__(self, idx):
        img = Image.fromarray(self.images[idx]).convert('RGB')
        img = TF.resize(img, (224, 224))
        orig = self.base_tf(img)
        pert = apply_perturbation(img, self.p_type, transforms.ToTensor(), self.norm_tf)
        return torch.stack([orig, pert]), idx


class FASSImageNetDataset(Dataset):
    def __init__(self, dataframe, p_type):
        self.df = dataframe
        self.p_type = p_type
        self.base_tf = transforms.Compose([transforms.ToTensor(), transforms.Normalize(NORM_MEAN, NORM_STD)])
        self.norm_tf = transforms.Normalize(NORM_MEAN, NORM_STD)
    def __len__(self): return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_data = row['image_bytes']
        if isinstance(img_data, dict):
            img = Image.open(BytesIO(img_data['bytes'])).convert('RGB')
        elif isinstance(img_data, bytes):
            img = Image.open(BytesIO(img_data)).convert('RGB')
        else:
            img = img_data.convert('RGB')
        img = TF.resize(img, (224, 224))
        orig = self.base_tf(img)
        pert = apply_perturbation(img, self.p_type, transforms.ToTensor(), self.norm_tf)
        return torch.stack([orig, pert]), idx


class FASSCOCODataset(Dataset):
    def __init__(self, root, image_list, p_type):
        self.root = root
        self.image_list = image_list
        self.p_type = p_type
        self.base_tf = transforms.Compose([transforms.ToTensor(), transforms.Normalize(NORM_MEAN, NORM_STD)])
        self.norm_tf = transforms.Normalize(NORM_MEAN, NORM_STD)
    def __len__(self): return len(self.image_list)
    def __getitem__(self, idx):
        img = Image.open(os.path.join(self.root, self.image_list[idx])).convert('RGB')
        img = TF.resize(img, (224, 224))
        orig = self.base_tf(img)
        pert = apply_perturbation(img, self.p_type, transforms.ToTensor(), self.norm_tf)
        return torch.stack([orig, pert]), idx

In [ ]:
def prepare_models(num_classes):
    models_dict = {
        'resnet50': models.resnet50(weights='IMAGENET1K_V1'),
        'densenet121': models.densenet121(weights='IMAGENET1K_V1'),
        'convnext_tiny': models.convnext_tiny(weights='IMAGENET1K_V1'),
        'vit_b_16': models.vit_b_16(weights='IMAGENET1K_V1'),
    }
    if num_classes != 1000:
        for name, m in models_dict.items():
            if 'resnet' in name: m.fc = nn.Linear(m.fc.in_features, num_classes)
            elif 'dense' in name: m.classifier = nn.Linear(m.classifier.in_features, num_classes)
            elif 'convnext' in name: m.classifier[2] = nn.Linear(m.classifier[2].in_features, num_classes)
            elif 'vit' in name: m.heads.head = nn.Linear(m.heads.head.in_features, num_classes)
    for m in models_dict.values():
        for module in m.modules():
            if isinstance(module, nn.ReLU): module.inplace = False
        m.eval()
    return models_dict


def get_gradcam_layer(model, name):
    if name == 'resnet50': return model.layer4[-1]
    elif name == 'densenet121': return model.features[-1]
    elif name == 'convnext_tiny': return model.features[-1]
    elif name == 'vit_b_16': return model.encoder.layers[-1]


def load_dataset(ds_name):
    cfg = DATASETS[ds_name]
    if cfg['type'] == 'npz':
        data = np.load(cfg['data_path'])
        for key in ['images', 'x', 'data']:
            if key in data:
                return data[key][:MAX_IMAGES], min(len(data[key]), MAX_IMAGES)
        first_key = [k for k in data.keys() if data[k].ndim == 4][0]
        return data[first_key][:MAX_IMAGES], min(len(data[first_key]), MAX_IMAGES)
    elif cfg['type'] == 'parquet':
        df = pd.read_parquet(cfg['data_path']).head(MAX_IMAGES)
        return df, len(df)
    elif cfg['type'] == 'directory':
        img_list = sorted([f for f in os.listdir(cfg['data_path']) if f.lower().endswith(('.jpg', '.jpeg', '.png'))])[:MAX_IMAGES]
        return img_list, len(img_list)


def make_dataloader(ds_name, data_obj, p_type, start_idx=0):
    cfg = DATASETS[ds_name]
    if cfg['type'] == 'npz':
        ds = FASSCIFAR10Dataset(data_obj[start_idx:], p_type)
    elif cfg['type'] == 'parquet':
        ds = FASSImageNetDataset(data_obj.iloc[start_idx:].reset_index(drop=True), p_type)
    elif cfg['type'] == 'directory':
        ds = FASSCOCODataset(cfg['data_path'], data_obj[start_idx:], p_type)
    return DataLoader(ds, batch_size=BATCH_SIZE, num_workers=NUM_WORKERS, pin_memory=True, drop_last=False)

In [ ]:
def save_checkpoint(results, path):
    df = pd.DataFrame(results)
    write_header = not os.path.exists(path)
    df.to_csv(path, mode='a', header=write_header, index=False)


def run_fast_benchmark(model_name, model, ds_name, data_obj, total_images, p_type):
    result_dir = DATASETS[ds_name]['result_dir']
    path = os.path.join(result_dir, f'{model_name}_fass_{p_type}.csv')

    start = 0
    if os.path.exists(path):
        try:
            existing = pd.read_csv(path)
            if not existing.empty:
                start = existing['image_index'].max() + 1
        except Exception:
            pass

    if start >= total_images:
        print(f'    {model_name} / {p_type} -- already complete, skipping')
        return

    model.to(DEVICE)
    ig = IntegratedGradients(model)
    gs = GradientShap(model)
    gcam = LayerGradCam(model, get_gradcam_layer(model, model_name))
    loader = make_dataloader(ds_name, data_obj, p_type, start)
    results = []

    pbar = tqdm(total=total_images - start, desc=f'    {model_name} / {p_type}', unit='img')

    for stack, idxs in loader:
        B = stack.shape[0]
        orig = stack[:, 0].to(DEVICE).requires_grad_(True)
        pert = stack[:, 1].to(DEVICE)

        with torch.no_grad():
            pred_orig = torch.argmax(model(orig), dim=1)
            pred_pert = torch.argmax(model(pert), dim=1)
            mask = (pred_orig == pred_pert)

        pbar.update(B)

        if not mask.any():
            continue

        o = orig[mask].requires_grad_(True)
        p = pert[mask].requires_grad_(True)
        t = pred_orig[mask]
        idx_pass = idxs[mask.cpu()]

        try:
            fass_ig = calculate_fass(ig.attribute(o, target=t), ig.attribute(p, target=t))
            fass_gs = calculate_fass(
                gs.attribute(o, baselines=torch.zeros_like(o), target=t, n_samples=5),
                gs.attribute(p, baselines=torch.zeros_like(p), target=t, n_samples=5))
            attr_gc_o = TF.resize(gcam.attribute(o, target=t), (224, 224))
            attr_gc_p = TF.resize(gcam.attribute(p, target=t), (224, 224))
            fass_gc = calculate_fass(attr_gc_o, attr_gc_p)

            for i in range(mask.sum().item()):
                results.append({
                    'image_index': idx_pass[i].item() + start,
                    'ig_fass': fass_ig[i].item(),
                    'shap_fass': fass_gs[i].item(),
                    'gradcam_fass': fass_gc[i].item(),
                })
        except RuntimeError:
            torch.cuda.empty_cache()
            continue

        if len(results) >= CHECKPOINT_EVERY:
            save_checkpoint(results, path)
            results = []

        torch.cuda.empty_cache()

    if results:
        save_checkpoint(results, path)

    pbar.close()
    print(f'    {model_name} / {p_type} -- all images processed')

    del ig, gs, gcam
    torch.cuda.empty_cache()

In [ ]:
for ds_name, cfg in DATASETS.items():
    print(f'\n=== {ds_name.upper()} ===')
    data_obj, total = load_dataset(ds_name)
    total = min(total, MAX_IMAGES)
    models_dict = prepare_models(cfg['num_classes'])

    for model_name in MODEL_NAMES:
        model = models_dict[model_name].to(DEVICE)
        for p_type in PERTURBATIONS_TO_RUN:
            run_fast_benchmark(model_name, model, ds_name, data_obj, total, p_type)
        model.cpu()
        del model
        torch.cuda.empty_cache()
        gc.collect()

    del models_dict, data_obj
    gc.collect()
    torch.cuda.empty_cache()

print('\nAll done.')

In [ ]:
ALL_PERTS = ['rotation', 'noise', 'translation', 'brightness', 'jpeg']

for ds_name, cfg in DATASETS.items():
    print(f'\n{ds_name.upper()}:')
    for model_name in MODEL_NAMES:
        for p_type in ALL_PERTS:
            path = os.path.join(cfg['result_dir'], f'{model_name}_fass_{p_type}.csv')
            if os.path.exists(path):
                df = pd.read_csv(path)
                cols = [c for c in df.columns if '_fass' in c]
                means = ' | '.join([f'{c.replace("_fass","")}: {df[c].mean():.4f}' for c in cols])
                print(f'  {model_name:17s} {p_type:12s} {len(df):6d} rows  {means}')
            else:
                print(f'  {model_name:17s} {p_type:12s} MISSING')